## Compact summary table of this notebook findings 

In [17]:
summary = pd.DataFrame([
    {"metric": "rows_in_exact_raw_duplicates", "value": n_raw_dup_rows},
    {"metric": "unique_exact_raw_duplicated_texts", "value": n_raw_dup_unique_texts},
    {"metric": "rows_in_exact_(raw_text,label)_duplicates", "value": n_raw_label_dup_rows},
    {"metric": "raw_texts_with_conflicting_labels", "value": n_conflicting_raw},
    {"metric": "rows_in_preprocessed_duplicates", "value": n_prep_dup_rows},
    {"metric": "unique_preprocessed_duplicated_texts", "value": n_prep_dup_unique},
    {"metric": "rows_in_exact_(preprocessed_text,label)_duplicates", "value": n_prep_label_dup_rows},
    {"metric": "preprocessed_texts_with_conflicting_labels", "value": n_conflicting_prep},
])

display(summary)

,metric,value
0,rows_in_exact_raw_duplicates,150
1,unique_exact_raw_duplicated_texts,68
2,"rows_in_exact_(raw_text,label)_duplicates",137
3,raw_texts_with_conflicting_labels,7
4,rows_in_preprocessed_duplicates,336
5,unique_preprocessed_duplicated_texts,147
6,"rows_in_exact_(preprocessed_text,label)_duplicates",288
7,preprocessed_texts_with_conflicting_labels,27


## Interpretation and Conclusion

### What each metric means

- **`rows_in_exact_raw_duplicates = 150`**  
  Number of rows whose **raw text** appears at least twice in the dataset.  
  If one exact review appears 4 times, then these 4 rows are all counted here.

- **`unique_exact_raw_duplicated_texts = 68`**  
  Number of **distinct raw texts** that are duplicated.  
  So, 68 different review strings appear more than once.

- **`rows_in_exact_(raw_text,label)_duplicates = 137`**  
  Number of rows that belong to duplicate groups where **both the raw text and the label are identical**.  
  This captures the “non-problematic” duplicates, i.e. repeated examples that are perfectly consistent.

- **`raw_texts_with_conflicting_labels = 7`**  
  Number of **distinct raw texts** that appear with **more than one label**.  
  This is important because it suggests either annotation inconsistency or ambiguity in the labeling process.

- **`rows_in_preprocessed_duplicates = 336`**  
  Number of rows whose **preprocessed text** appears at least twice.  
  This is computed after normalization / cleaning, so different raw texts may collapse to the same canonical form.

- **`unique_preprocessed_duplicated_texts = 147`**  
  Number of **distinct preprocessed texts** that are duplicated.

- **`rows_in_exact_(preprocessed_text,label)_duplicates = 288`**  
  Number of rows that are duplicated after preprocessing **and** still share the same label.

- **`preprocessed_texts_with_conflicting_labels = 27`**  
  Number of **distinct preprocessed texts** that appear with **multiple labels** after preprocessing.

---


#### 1. Raw duplicates are present, but not extreme
We find:
- **150 rows** involved in exact raw duplicates
- corresponding to **68 distinct duplicated raw texts**

This means the dataset contains a **non-negligible amount of redundancy** at the raw text level.  
The average duplicated raw text appears a little more than twice on average, which is plausible for short, templated reviews such as:
- “Exactly as described.”
- “Works as advertised.”
- “Never received the item.”
- “Erfüllt seinen Zweck.”

So i guess que ca fait du sens.

#### 2. Most raw duplicates are label-consistent
Among the 150 rows involved in raw duplicates, **137 rows** are duplicated together with the **same label**.

This suggests that **most repeated reviews are not contradictory**: the same exact text is usually assigned the same sentiment / class.  
That is reassuring, because it means duplicates are mostly simple redundancy rather than obvious label corruption.

#### 3. But there are still some raw-text label conflicts
We also observe **7 distinct raw texts with conflicting labels**.


#### 4. Preprocessing creates many more collisions
After preprocessing, the numbers increase strongly:

- duplicated rows: **150 → 336**
- duplicated unique texts: **68 → 147**
- conflicting duplicated texts: **7 → 27**

This is also fully logical.

Preprocessing removes or normalizes parts of the surface form:
- punctuation
- formatting
- possibly some noisy tokens
- case differences
- other textual variations

As a result, **different raw reviews can collapse to the same preprocessed representation**.  
For example, texts like:

- `Three Stars\n\nSmaller than expected`
- `Three Stars\n\nSmaller than expected.`
- `Three Stars\n\nSmaller than I expected`

may become much more similar, or even identical, after cleaning.

So the rise in duplicates after preprocessing is not a bug: it is exactly what we would expect from an aggressive normalization pipeline.

#### 5. Preprocessing also increases label conflicts
The jump from:
- **7 conflicting raw texts**
to
- **27 conflicting preprocessed texts**

is particularly important.

This means preprocessing is not only merging harmless surface variants it is also **collapsing some texts that do not always share the same label**.


---

### Main interpretation

Overall, the duplicate analysis suggests three key points:

1. **The dataset contains noticeable redundancy**  
   A number of reviews are repeated verbatim, especially short and generic ones.

2. **Most exact duplicates are label-consistent**  
   So duplication is mostly a redundancy issue rather than a major annotation failure.

3. **Preprocessing substantially increases text collisions and label conflicts**  
   This suggests that normalization may be collapsing semantically or sentiment-relevant distinctions.



# Analysis on Duplicates

This notebook continues the previous exploratory notebook and focuses on:
- loading the training data
- reusing existing preprocessing when available
- exact duplicates
- duplicates after preprocessing
- conflicting labels for identical text
- short data-quality summary

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 100)

current_dir = Path.cwd()

if current_dir.name == "data_analysis":
    REPO_ROOT = current_dir.parent
else:
    REPO_ROOT = current_dir

DATA_DIR = REPO_ROOT / "data"
ANALYSIS_DIR = REPO_ROOT / "data_analysis"

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

if str(ANALYSIS_DIR) not in sys.path:
    sys.path.append(str(ANALYSIS_DIR))

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR:", DATA_DIR)
print("ANALYSIS_DIR:", ANALYSIS_DIR)

REPO_ROOT: /Users/yousseftorgeman/Desktop/text_classifier
DATA_DIR: /Users/yousseftorgeman/Desktop/text_classifier/data
ANALYSIS_DIR: /Users/yousseftorgeman/Desktop/text_classifier/data_analysis


In [2]:
from data_analysis.utils.text_preprocessing import TextPreprocessor

## 1. Load training data

In [3]:
train_path = DATA_DIR / "train.csv"
assert train_path.exists(), f"Missing file: {train_path}"

train_df = pd.read_csv(train_path)

print(f"Train shape: {train_df.shape}")
display(train_df.head())
print("Columns:", train_df.columns.tolist())

Train shape: (252000, 3)


,id,sentence,label
0,0,"Moderner Weihnachtsbaum in weiß\n\nDieses Jahr wollte ich keinen echten Baum und so hab ich mich für einen modernen kleinen in weiß entschieden. Als das Paket kam und ich den Baum ausgepackt hab, ...",4
1,1,"Passt wie angegossen\n\nTasche kam schnell und gut Verpackt an, die kleine DJI Ryze Tello passt perfekt rein sowie der Controller , zusätzlich sind noch 2 schlitze für Reserve Akkus und 3 Vertiefu...",4
2,2,Schlechte Qualität\n\nIch habe sehr lange auf diesen Käfig gespart. Leider muss ich mich den vorherigen Rezensionen anschließen. Die Qualität ist wirklich sehr schlecht. Überall ist das Holz gespl...,1
3,3,Bestellung nie angekommen\n\n-5 Sterne..... Am 15.5 bestellt und bis jetzt immer noch nicht da. Das Teil können Sie behalten,0
4,4,"Für mich gar nicht gut\n\nMacht das Makeup gar nicht haltbarer. Hinterlässt ein klebriges Gefühl auf der Haut, was gar nicht verschwindet. Sprühkopf verteilt das Produkt überhaupt nicht gut",1


Columns: ['id', 'sentence', 'label']


In [4]:
TEXT_COL = "sentence"
LABEL_COL = "label"

assert TEXT_COL in train_df.columns, f"Column '{TEXT_COL}' not found in train.csv"
assert LABEL_COL in train_df.columns, f"Column '{LABEL_COL}' not found in train.csv"

train_df[TEXT_COL] = train_df[TEXT_COL].fillna("").astype(str).str.strip()

print("Number of empty texts:", (train_df[TEXT_COL] == "").sum())
print("Labels:", sorted(train_df[LABEL_COL].dropna().unique()))

Number of empty texts: 0
Labels: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


## 2. Use of sacha preprocessing utils 


In [ ]:
splitlang_debug_path = DATA_DIR / "splitlang" / "DEBUG" / "train.csv"
ner_debug_path = DATA_DIR / "splitlang" / "DEBUG" / "ner_changes_train.csv"

debug_df = None
ner_df = None

if splitlang_debug_path.exists():
    debug_df = pd.read_csv(splitlang_debug_path)
    print(f"Loaded debug_df: {debug_df.shape}")
else:
    print("DEBUG split file not xfound. Will preprocess locally.")

if ner_debug_path.exists():
    ner_df = pd.read_csv(ner_debug_path)
    print(f"Loaded ner_df: {ner_df.shape}")
else:
    print("NER changes file not found.")

DEBUG split file not found. Will preprocess locally.
NER changes file not found.


In [18]:
PREPROCESSED_COL = "_preprocessed_text"

print("Generating preprocessed text directly with the utils direct.")

preprocessor = TextPreprocessor()
train_df[TEXT_COL] = train_df[TEXT_COL].fillna("").astype(str)

train_df[PREPROCESSED_COL] = train_df[TEXT_COL].apply(preprocessor.preprocess)

print("Preprocessing done.")
print("Empty preprocessed texts:", (train_df[PREPROCESSED_COL].astype(str).str.strip() == "").sum())

display(train_df[[TEXT_COL, PREPROCESSED_COL, LABEL_COL]].head())

Generating preprocessed text directly with the utils direct.
Preprocessing done.
Empty preprocessed texts: 0


,sentence,_preprocessed_text,label
0,"Moderner Weihnachtsbaum in weiß\n\nDieses Jahr wollte ich keinen echten Baum und so hab ich mich für einen modernen kleinen in weiß entschieden. Als das Paket kam und ich den Baum ausgepackt hab, ...",moderner weihnachtsbaum in weiß dieses jahr wollte ich keinen echten baum und so hab ich mich für einen modernen kleinen in weiß entschieden als das paket kam und ich den baum ausgepackt hab dacht...,4
1,"Passt wie angegossen\n\nTasche kam schnell und gut Verpackt an, die kleine DJI Ryze Tello passt perfekt rein sowie der Controller , zusätzlich sind noch 2 schlitze für Reserve Akkus und 3 Vertiefu...",passt wie angegossen tasche kam schnell und gut verpackt an die kleine dji ryze tello passt perfekt rein sowie der controller zusätzlich sind noch 2 schlitze für reserve akkus und 3 vertiefungen f...,4
2,Schlechte Qualität\n\nIch habe sehr lange auf diesen Käfig gespart. Leider muss ich mich den vorherigen Rezensionen anschließen. Die Qualität ist wirklich sehr schlecht. Überall ist das Holz gespl...,schlechte qualität ich habe sehr lange auf diesen käfig gespart leider muss ich mich den vorherigen rezensionen anschließen die qualität ist wirklich sehr schlecht überall ist das holz gesplittert...,1
3,Bestellung nie angekommen\n\n-5 Sterne..... Am 15.5 bestellt und bis jetzt immer noch nicht da. Das Teil können Sie behalten,bestellung nie angekommen 5 sterne am 15 5 bestellt und bis jetzt immer noch nicht da das teil können sie behalten,0
4,"Für mich gar nicht gut\n\nMacht das Makeup gar nicht haltbarer. Hinterlässt ein klebriges Gefühl auf der Haut, was gar nicht verschwindet. Sprühkopf verteilt das Produkt überhaupt nicht gut",für mich gar nicht gut macht das makeup gar nicht haltbarer hinterlässt ein klebriges gefühl auf der haut was gar nicht verschwindet sprühkopf verteilt das produkt überhaupt nicht gut,1


In [19]:
def print_section(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

## 3. Exact duplicates on raw text

In [20]:
print_section("3.1 Exact duplicates on raw text")

raw_dup_mask = train_df.duplicated(subset=[TEXT_COL], keep=False)
raw_dup_df = train_df.loc[raw_dup_mask].copy()

n_raw_dup_rows = int(raw_dup_mask.sum())
n_raw_dup_unique_texts = int(raw_dup_df[TEXT_COL].nunique()) if len(raw_dup_df) > 0 else 0

print(f"Rows involved in exact raw duplicates: {n_raw_dup_rows}")
print(f"Unique duplicated raw texts: {n_raw_dup_unique_texts}")

raw_dup_counts = (
    raw_dup_df.groupby(TEXT_COL)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(raw_dup_counts.head(20))


3.1 Exact duplicates on raw text
Rows involved in exact raw duplicates: 150
Unique duplicated raw texts: 68


,sentence,count
51,Three Stars\n\nSmaller than expected,4
14,Five Stars\n\nExactly as described.,4
8,Artikel nicht erhalten\n\nArtikel nicht erhalten,4
52,Three Stars\n\nSmaller than expected.,3
60,Two Stars\n\nSmaller than expected,3
26,Gut\n\nErfüllt seinen Zweck,3
49,Three Stars\n\nSmaller than I expected,3
22,Four Stars\n\nWorks as advertised.,3
20,Four Stars\n\nVery soft and comfortable,3
38,One Star\n\nNever received the item.,3


## 4. Exact duplicates on `(text, label)`

In [21]:
print_section("4.1 Exact duplicates on raw text WITH same label")

raw_label_dup_mask = train_df.duplicated(subset=[TEXT_COL, LABEL_COL], keep=False)
raw_label_dup_df = train_df.loc[raw_label_dup_mask].copy()

n_raw_label_dup_rows = int(len(raw_label_dup_df))

print(f"Rows involved in duplicated (text, label) pairs: {n_raw_label_dup_rows}")

raw_label_dup_counts = (
    raw_label_dup_df.groupby([TEXT_COL, LABEL_COL])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(raw_label_dup_counts.head(50))


4.1 Exact duplicates on raw text WITH same label
Rows involved in duplicated (text, label) pairs: 137


,sentence,label,count
46,Three Stars\n\nSmaller than expected,2,4
8,Artikel nicht erhalten\n\nArtikel nicht erhalten,0,4
13,Five Stars\n\nExactly as described.,4,4
19,Four Stars\n\nVery soft and comfortable,3,3
35,One Star\n\nNever received the item.,0,3
47,Three Stars\n\nSmaller than expected.,2,3
57,nur zu empfehlen\n\ngut Produkt wie vorgestellt. einfach erfühlt sein Aufgabe. Lieferung pünktlich. Verpackung gut preis passt . allgemein bin ich sehr zufrieden mit dem Artikel.,3,3
21,Four Stars\n\nWorks as advertised.,3,3
55,Two Stars\n\nSmaller than expected,1,3
44,Three Stars\n\nSmaller than I expected,2,3


## 5. Conflicting labels for identical raw text

In [22]:
print_section("5.1 Conflicting labels for identical raw text")

conflicting_raw = (
    train_df.groupby(TEXT_COL)[LABEL_COL]
    .nunique()
    .reset_index(name="n_unique_labels")
)

conflicting_raw = conflicting_raw[conflicting_raw["n_unique_labels"] > 1].copy()

n_conflicting_raw = int(len(conflicting_raw))
print(f"Raw texts associated with multiple labels: {n_conflicting_raw}")

if n_conflicting_raw > 0:
    conflicting_raw_examples = train_df.merge(conflicting_raw[[TEXT_COL]], on=TEXT_COL, how="inner")
    conflicting_raw_examples = conflicting_raw_examples.sort_values([TEXT_COL, LABEL_COL])
    display(conflicting_raw_examples[[TEXT_COL, LABEL_COL]].head(100))
else:
    print("No conflicting raw-text labels found.")


5.1 Conflicting labels for identical raw text
Raw texts associated with multiple labels: 7


,sentence,label
5,Erfüllt seinen Zweck\n\nErfüllt seinen Zweck,2
0,Erfüllt seinen Zweck\n\nErfüllt seinen Zweck,3
6,Gut\n\nErfüllt seinen Zweck,3
10,Gut\n\nErfüllt seinen Zweck,3
2,Gut\n\nErfüllt seinen Zweck,4
4,Keine Rechnung erhalten\n\nKeine Rechnung erhalten,0
12,Keine Rechnung erhalten\n\nKeine Rechnung erhalten,3
7,Ok\n\nErfüllt seinen Zweck,2
9,Ok\n\nErfüllt seinen Zweck,3
3,Passt\n\nErfüllt seinen Zweck,3


## 6. Duplicates after preprocessing

In [23]:
print_section("6.1 Duplicates after preprocessing")

prep_dup_mask = train_df.duplicated(subset=[PREPROCESSED_COL], keep=False)
prep_dup_df = train_df.loc[prep_dup_mask].copy()

n_prep_dup_rows = int(prep_dup_mask.sum())
n_prep_dup_unique = int(prep_dup_df[PREPROCESSED_COL].nunique()) if len(prep_dup_df) > 0 else 0

print(f"Rows involved in duplicates after preprocessing: {n_prep_dup_rows}")
print(f"Unique duplicated preprocessed texts: {n_prep_dup_unique}")

prep_dup_counts = (
    prep_dup_df.groupby(PREPROCESSED_COL)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(prep_dup_counts.head(20))


6.1 Duplicates after preprocessing
Rows involved in duplicates after preprocessing: 336
Unique duplicated preprocessed texts: 147


,_preprocessed_text,count
121,three stars smaller than expected,9
2,alles bestens gerne wieder alles bestens gerne wieder,7
22,five stars exactly as described,5
10,artikel nicht erhalten artikel nicht erhalten,5
122,three stars smaller than i expected,4
90,one star never received the item,4
32,five stars works as advertised,3
59,gut erfüllt seinen zweck,3
118,three stars have not used it yet,3
127,top alles wie beschrieben,3


## 7. Exact duplicates on `(preprocessed_text, label)`

In [24]:
print_section("7.1 Duplicates after preprocessing WITH same label")

prep_label_dup_mask = train_df.duplicated(subset=[PREPROCESSED_COL, LABEL_COL], keep=False)
prep_label_dup_df = train_df.loc[prep_label_dup_mask].copy()

n_prep_label_dup_rows = int(len(prep_label_dup_df))
print(f"Rows involved in duplicated (preprocessed_text, label) pairs: {n_prep_label_dup_rows}")

prep_label_dup_counts = (
    prep_label_dup_df.groupby([PREPROCESSED_COL, LABEL_COL])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(prep_label_dup_counts.head(20))


7.1 Duplicates after preprocessing WITH same label
Rows involved in duplicated (preprocessed_text, label) pairs: 288


,_preprocessed_text,label,count
104,three stars smaller than expected,2,9
2,alles bestens gerne wieder alles bestens gerne wieder,4,6
20,five stars exactly as described,4,5
10,artikel nicht erhalten artikel nicht erhalten,0,5
105,three stars smaller than i expected,2,4
82,one star never received the item,0,4
78,one star i never received this item,0,3
44,four stars works as advertised,3,3
32,four stars as expected thank you,3,3
101,three stars have not used it yet,2,3


## 8. Conflicting labels after preprocessing

In [25]:
print_section("8.1 Conflicting labels for identical preprocessed text")

conflicting_prep = (
    train_df.groupby(PREPROCESSED_COL)[LABEL_COL]
    .nunique()
    .reset_index(name="n_unique_labels")
)

conflicting_prep = conflicting_prep[conflicting_prep["n_unique_labels"] > 1].copy()

n_conflicting_prep = int(len(conflicting_prep))
print(f"Preprocessed texts associated with multiple labels: {n_conflicting_prep}")

if n_conflicting_prep > 0:
    conflicting_prep_examples = train_df.merge(conflicting_prep[[PREPROCESSED_COL]], on=PREPROCESSED_COL, how="inner")
    conflicting_prep_examples = conflicting_prep_examples.sort_values([PREPROCESSED_COL, LABEL_COL])
    display(conflicting_prep_examples[[TEXT_COL, PREPROCESSED_COL, LABEL_COL]].head(40))
else:
    print("No conflicting preprocessed-text labels found.")


8.1 Conflicting labels for identical preprocessed text
Preprocessed texts associated with multiple labels: 27


,sentence,_preprocessed_text,label
54,"Alles Bestens, gerne wieder\n\nAlles Bestens, gerne wieder",alles bestens gerne wieder alles bestens gerne wieder,3
0,"alles bestens, gerne wieder\n\nalles bestens, gerne wieder",alles bestens gerne wieder alles bestens gerne wieder,4
18,"Alles bestens, gerne wieder.\n\nAlles bestens, gerne wieder.",alles bestens gerne wieder alles bestens gerne wieder,4
19,"Alles bestens, gerne wieder.\n\nAlles bestens, gerne wieder.",alles bestens gerne wieder alles bestens gerne wieder,4
28,Alles bestens - gerne wieder\n\nAlles bestens - gerne wieder,alles bestens gerne wieder alles bestens gerne wieder,4
52,Alles Bestens gerne wieder\n\nAlles Bestens gerne wieder,alles bestens gerne wieder alles bestens gerne wieder,4
62,Alles bestens - gerne wieder\n\nAlles bestens - gerne wieder,alles bestens gerne wieder alles bestens gerne wieder,4
32,Alles wie beschrieben\n\nAlles wie beschrieben,alles wie beschrieben alles wie beschrieben,3
49,Alles wie beschrieben\n\nAlles wie beschrieben!,alles wie beschrieben alles wie beschrieben,3
39,alles wie beschrieben\n\nalles wie beschrieben,alles wie beschrieben alles wie beschrieben,4


## 9. Manual inspection of strongest duplicate patterns

In [26]:
print_section("9.1 Top duplicated raw texts")

if len(raw_dup_counts) > 0:
    top_raw_examples = raw_dup_counts.head(10).merge(train_df, on=TEXT_COL, how="left")
    display(top_raw_examples[[TEXT_COL, LABEL_COL, "count"]].sort_values("count", ascending=False).head(30))
else:
    print("No raw duplicates found.")


9.1 Top duplicated raw texts


,sentence,label,count
0,Three Stars\n\nSmaller than expected,2,4
7,Five Stars\n\nExactly as described.,4,4
1,Three Stars\n\nSmaller than expected,2,4
10,Artikel nicht erhalten\n\nArtikel nicht erhalten,0,4
9,Artikel nicht erhalten\n\nArtikel nicht erhalten,0,4
8,Artikel nicht erhalten\n\nArtikel nicht erhalten,0,4
11,Artikel nicht erhalten\n\nArtikel nicht erhalten,0,4
6,Five Stars\n\nExactly as described.,4,4
5,Five Stars\n\nExactly as described.,4,4
4,Five Stars\n\nExactly as described.,4,4
